In [1]:
import importlib
import mixed_data
importlib.reload(mixed_data)

tokens = mixed_data.load_shakespeare_tokens()
next_ex = mixed_data.make_next_examples(tokens=tokens[:80])
fill_ex = mixed_data.make_fill_examples(tokens=tokens[:80])

print("shakespeare tokens", len(tokens))
print("next example", next_ex[0])
print("fill example", fill_ex[0])
print("n next", len(next_ex), "n fill", len(fill_ex))
assert next_ex[0][0] == "[next]"
assert fill_ex[0][0] == "[fill]"
print("next/fill builders ok")

shakespeare tokens 250148
next example ['[next]', 'first', 'citizen', ':', 'before']
fill example ['[fill]', 'first', '[mask]', ':', 'before', '[sep]', 'citizen']
n next 41 n fill 41
next/fill builders ok


In [2]:
import importlib
import mixed_data
importlib.reload(mixed_data)

bitext = mixed_data.make_translate_examples()
print("n bitext", len(bitext))
print("first", bitext[0])
print("second", bitext[1])
assert bitext[0][0] == "[en2ne]"
assert bitext[1][0] == "[ne2en]"
assert "[sep]" in bitext[0]
print("translate builders ok")

n bitext 40
first ['[en2ne]', 'hello', '[sep]', 'नमस्ते']
second ['[ne2en]', 'नमस्ते', '[sep]', 'hello']
translate builders ok


In [3]:
import importlib
import mixed_data
importlib.reload(mixed_data)

loaders, vocab, stats = mixed_data.build_dataloaders(batch_size=8)
print(stats)
batch = next(iter(loaders["train"]))
print("input", tuple(batch["input_ids"].shape))
print("decoded first:", vocab.decode(batch["input_ids"][0].tolist())[:12])
assert "[next]" in vocab.stoi and "[en2ne]" in vocab.stoi
print("mixed loader ok")

{'n_train': 1056, 'n_val': 104, 'n_test': 104, 'n_next_train': 400, 'n_fill_train': 400, 'n_bitext_train': 256, 'vocab_size': 968}
input (8, 16)
decoded first: ['[ne2en]', 'धन्यवाद', '[sep]', 'thank', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
mixed loader ok


In [4]:
import torch
import mixed_data
from model import NextWordModel

loaders, vocab, stats = mixed_data.build_dataloaders(batch_size=4)
batch = next(iter(loaders["train"]))

net = NextWordModel(
    vocab_size=stats["vocab_size"],
    embed_size=64,
    max_len=48,
    pad_id=vocab.pad_id(),
    dropout=0.0,
)
net.eval()

with torch.no_grad():
    logits = net.forward(input_ids=batch["input_ids"])

print("input ", tuple(batch["input_ids"].shape))
print("logits", tuple(logits.shape))
assert logits.shape[0] == batch["input_ids"].shape[0]
assert logits.shape[1] == batch["input_ids"].shape[1]
assert logits.shape[2] == stats["vocab_size"]
print("Q3 model test ok")

input  (4, 14)
logits (4, 14, 968)
Q3 model test ok
